# 04 — Matched-Compute MLM Control (`04_matched_compute_mlm_control.ipynb`)

**Hypothesis Tested:** Does the performance gain of `MLM+attn` over `noMLM+attn` (+0.055 4-class Sentiment Macro-F1 on Run A, +0.024 on Run B) come from **in-domain language adaptation** on code-mixed Sinhala customer care text, or merely from **extra gradient updates / encoder optimization steps**?

We test two matched-compute controls using the exact same splits, class weights, attention pooling architecture, and seeds (`42, 43, 44`):
1. **Variant X (`genericMLM+attn` — Non-Domain / Shuffled-Context MLM Control):**
   - Runs 3 epochs of MLM with identical hyperparameters (`MLM_PROB=0.15`, `MLM_LR=2e-5`, `MLM_BATCH=8`, `MLM_GRAD_ACC=2`) on an equal-sized **non-domain control corpus** (either an external generic Sinhala news/Wikipedia CSV if provided in `GENERIC_SINHALA_CSV`, or a cross-sentence random token-salad permutation of the corpus that preserves exact unigram token frequencies and step count while destroying all customer-care phrase/collocation structure), then fine-tunes with task-specific attention pooling (`attn`).
2. **Variant Y (`noMLM+attn_matched_compute` — Extra Fine-Tuning Compute / Higher Encoder LR Control):**
   - `Variant Y1` (`noMLM+attn_13ep`): Trains `noMLM+attn` for `13` supervised epochs (matching the 3 MLM + 10 FT epoch budget) with increased early-stopping patience.
   - `Variant Y2` (`noMLM+attn_highEncLR`): Trains `noMLM+attn` with the encoder learning rate raised to `2e-5` (`ENC_LR_MULT = 1.0`, matching `MLM_LR`).

## 1. Environment & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import gc
import math
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    NS,
    NI,
    SEEDS,
    log_environment,
    set_all_seeds,
    load_splits,
    clean_text,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# ── Single Config Cell ────────────────────────────────────────────────────────
MODEL_NAME          = "xlm-roberta-base"
MAX_LEN             = 128
MLM_PROB            = 0.15
MLM_EPOCHS          = 3
MLM_BATCH           = 8
MLM_GRAD_ACC        = 2
MLM_LR              = 2e-5

FT_EPOCHS           = 10
FT_BATCH            = 8
FT_GRAD_ACC         = 2
FT_LR               = 2e-5
ENC_LR_MULT         = 0.1
WEIGHT_DECAY        = 0.01
DROPOUT             = 0.3
NUM_ATTN_HEADS      = 4
PATIENCE            = 3

GENERIC_SINHALA_CSV = None
GENERIC_TEXT_COL    = "Sentence"

RUNS_TO_EXEC        = ["runA", "runB"]
CONTROL_VARIANTS    = [
    "genericMLM+attn",           # Variant X: 3 ep MLM on non-domain/shuffled text -> FT with attn
    "noMLM+attn_highEncLR",      # Variant Y: noMLM+attn with encoder LR = 2e-5 (matching MLM LR)
    "noMLM+attn_13ep",           # Variant Y2: noMLM+attn with 13 FT epochs (matching 3 MLM + 10 FT epochs)
]

WORK_DIR            = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV          = None
GOLD_CSV            = None
RESULTS_DIR         = os.path.join(WORK_DIR, "results")
PREDS_DIR           = os.path.join(WORK_DIR, "preds")
ENC_DIR             = os.path.join(WORK_DIR, "mlm_encoders_control")
for d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Non-Domain Control Corpus & Model Definitions

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_non_domain_mlm_examples(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, run: str, seed: int = 42):
    target_n = len(train_df) + len(val_df)
    domain_texts = train_df["text"].tolist() + val_df["text"].tolist()
    test_texts_set = set(test_df["text"].tolist())

    if GENERIC_SINHALA_CSV and os.path.exists(GENERIC_SINHALA_CSV):
        raw_gen = pd.read_csv(GENERIC_SINHALA_CSV, usecols=[GENERIC_TEXT_COL])[GENERIC_TEXT_COL].dropna()
        cleaned = raw_gen.map(lambda s: clean_text(s, run=run))
        cleaned = cleaned[cleaned.str.len() > 3].drop_duplicates()
        cleaned = cleaned[~cleaned.isin(test_texts_set | set(domain_texts))]
        ctrl_texts = cleaned.sample(min(target_n, len(cleaned)), random_state=seed).tolist()
        print(f"Loaded {len(ctrl_texts)} external generic non-domain texts for MLM control.")
    else:
        rng = random.Random(seed + 999)
        lengths = [len(t.split()) for t in domain_texts]
        all_words = [w for t in domain_texts for w in t.split()]
        rng.shuffle(all_words)
        ctrl_texts = []
        cursor = 0
        for L in lengths:
            L = max(1, L)
            chunk = all_words[cursor : cursor + L]
            cursor += L
            s = " ".join(chunk)
            if s in test_texts_set:
                s = s + " _"
            ctrl_texts.append(s)
        print(f"Constructed {len(ctrl_texts)} length-matched cross-sentence shuffled control texts (equal token & step budget).")

    assert not (set(ctrl_texts) & test_texts_set), "Leakage guard: test text must never appear in control MLM corpus"
    _e = tok(ctrl_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_e["input_ids"], _e["attention_mask"], _e["special_tokens_mask"])
    ]
    return examples

class DS(Dataset):
    def __init__(self, d):
        e = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids, self.am = e["input_ids"], e["attention_mask"]
        self.s, self.i = d["sent_id"].tolist(), d["intent_id"].tolist()
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        return {"input_ids": self.ids[k], "attention_mask": self.am[k], "sent": self.s[k], "intent": self.i[k]}

def collate(batch):
    out = tok.pad([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch], return_tensors="pt")
    out["sent"]   = torch.tensor([b["sent"] for b in batch])
    out["intent"] = torch.tensor([b["intent"] for b in batch])
    return out

class Pooler(nn.Module):
    def __init__(self, hidden, mode, heads, dropout):
        super().__init__()
        self.mode = mode
        if mode == "attn":
            self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
            self.query_sent = nn.Parameter(torch.randn(1, 1, hidden))
            self.query_int  = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        if self.mode == "attn":
            B, kpm = H.size(0), (mask == 0)
            outs = []
            for q in (self.query_sent, self.query_int):
                o, _ = self.attn(q.expand(B, -1, -1), H, H, key_padding_mask=kpm)
                outs.append(self.norm(o.squeeze(1)))
            return outs
        m = mask.unsqueeze(-1).to(H.dtype)
        p = self.norm((H * m).sum(1) / m.sum(1).clamp(min=1))
        return p, p

class JointModel(nn.Module):
    def __init__(self, enc_path, pooling="attn"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = Pooler(h, pooling, NUM_ATTN_HEADS, DROPOUT)
        fc = lambda: nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_fc, self.int_fc = fc(), fc()
        self.sent_head, self.intent_head = nn.Linear(h // 2, NS), nn.Linear(h // 2, NI)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ps, pi = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(ps)), self.intent_head(self.int_fc(pi))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    S, I = [], []
    for b in loader:
        s, i = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        S.append(torch.softmax(s, -1).cpu())
        I.append(torch.softmax(i, -1).cpu())
    return torch.cat(S).numpy(), torch.cat(I).numpy()

def class_weights(ids, K):
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

## 3. Control Training Pipeline (Variant X & Variant Y)

In [ ]:
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

def get_generic_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    out = os.path.join(ENC_DIR, f"generic_mlm_{run}_seed{seed}")
    if os.path.exists(os.path.join(out, "config.json")):
        return out
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)
    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item()
                n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
        print(f"  [Generic-MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")
    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def train_control_variant(variant_name: str, run: str, seed: int, train_df, val_df, test_df, mlm_examples):
    train_ds, val_ds, test_ds = DS(train_df), DS(val_df), DS(test_df)
    val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=FT_BATCH, shuffle=False, collate_fn=collate)

    W_SENT = class_weights(train_df["sent_id"].values, NS).to(DEVICE)
    W_INT  = class_weights(train_df["intent_id"].values, NI).to(DEVICE)
    val_ys, val_yi = val_df["sent_id"].values, val_df["intent_id"].values

    if variant_name == "genericMLM+attn":
        enc_path = get_generic_mlm_encoder(mlm_examples, run=run, seed=seed)
        enc_lr = FT_LR * ENC_LR_MULT
        epochs = FT_EPOCHS
        patience = PATIENCE
    elif variant_name == "noMLM+attn_highEncLR":
        enc_path = MODEL_NAME
        enc_lr = FT_LR
        epochs = FT_EPOCHS
        patience = PATIENCE
    elif variant_name == "noMLM+attn_13ep":
        enc_path = MODEL_NAME
        enc_lr = FT_LR * ENC_LR_MULT
        epochs = FT_EPOCHS + MLM_EPOCHS
        patience = PATIENCE + 2
    else:
        raise ValueError(f"Unknown control variant: {variant_name}")

    set_all_seeds(seed)
    model = JointModel(enc_path, pooling="attn").to(DEVICE)
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate, generator=g)
    crit_s, crit_i = nn.CrossEntropyLoss(weight=W_SENT), nn.CrossEntropyLoss(weight=W_INT)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": enc_lr},
         {"params": head_params, "lr": FT_LR}],
        weight_decay=WEIGHT_DECAY,
    )
    total = epochs * math.ceil(len(loader) / FT_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

    best_val, best_ep, best_state, bad = -1.0, 0, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        tl = 0.0
        opt.zero_grad()
        for step, b in enumerate(loader):
            s, i = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            loss = crit_s(s, b["sent"].to(DEVICE)) + crit_i(i, b["intent"].to(DEVICE))
            (loss / FT_GRAD_ACC).backward()
            tl += loss.item()
            if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
        vs, vi = predict_probs(model, val_loader)
        f_s = macro_f1_present(val_ys, vs.argmax(1), NS)
        f_i = macro_f1_present(val_yi, vi.argmax(1), NI)
        score = 0.5 * (f_s + f_i)
        print(f"  [{variant_name} {run} s{seed}] ep {ep+1:>2}/{epochs} | train {tl/len(loader):.4f} | val F1 sent {f_s:.3f} int {f_i:.3f} | mean {score:.3f}")
        if score > best_val:
            best_val, best_ep, bad = score, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print("  early stop")
                break

    model.load_state_dict(best_state)
    ts, ti = predict_probs(model, test_loader)
    del model, opt, sched, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    nb_tag = f"04_matched_compute_mlm_control_{variant_name}"
    payload = evaluate_and_save(
        notebook_name=nb_tag,
        model_name=variant_name,
        run=run,
        seed=seed,
        test_df=test_df,
        pred_sent_id=ts.argmax(1),
        pred_intent_id=ti.argmax(1),
        sent_probs=ts,
        intent_probs=ti,
        extra_meta={"best_epoch": best_ep, "best_val_mean_f1": best_val, "enc_lr": enc_lr, "max_epochs": epochs},
        results_dir=RESULTS_DIR,
        preds_dir=PREDS_DIR,
    )
    if variant_name == "genericMLM+attn":
        std_path = os.path.join(RESULTS_DIR, f"04_matched_compute_mlm_control_{run}_{seed}.json")
        with open(std_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
    return payload

## 4. Run All Seeds & Compare Against `MLM+attn` and `noMLM+attn`

In [ ]:
control_summaries = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_ctrl_examples = build_non_domain_mlm_examples(train_df, val_df, test_df, run=run, seed=42)

    for v_name in CONTROL_VARIANTS:
        seed_payloads = []
        for seed in SEEDS:
            p = train_control_variant(v_name, run, seed, train_df, val_df, test_df, mlm_ctrl_examples)
            seed_payloads.append(p)
        control_summaries.append(summarize_Across_seeds(seed_payloads))

display(pd.DataFrame(control_summaries)[["run", "model", "n_seeds", "sent_acc_str", "sent_f1_3cls_str", "sent_f1_4cls_str", "intent_acc_str", "intent_f1_str", "joint_acc_str"]])